In [ ]:
resources = "Patient,Encounter,Observation,Condition"

In [ ]:
RESOURCES = [r.strip() for r in resources.split(",") if r.strip()]

In [ ]:
from pyspark.sql import functions as F
from datetime import datetime, timezone

# --------------------------------------------------
# Configuration
# --------------------------------------------------



# --------------------------------------------------
# Gold layer: current, reporting-ready analytical tables
# --------------------------------------------------
for resource in RESOURCES:
    gold_table = f"gold_{resource.lower()}"

    current_df = (
        spark.table(f"silver_{resource.lower()}")
             .filter(F.col("is_current") == True)
    )

    (current_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(gold_table))

    print(f"Created {gold_table}")

# Child table for the repeating Patient.name array.
patient_names_df = (
    spark.table("silver_patient")
    .filter(F.col("is_current") == True)
    .select(
        F.col("id").alias("patient_id"),
        F.from_json(
            F.col("name"),
            "array<struct<use:string,family:string,given:array<string>,"
            "prefix:array<string>,suffix:array<string>,text:string>>"
        ).alias("name")
    )
    .select(
        "patient_id",
        F.posexplode_outer("name").alias("name_position", "name")
    )
    .select(
        "patient_id",
        "name_position",
        F.col("name.use").alias("name_use"),
        F.col("name.family").alias("family_name"),
        F.col("name.given").alias("given_names"),
        F.col("name.prefix").alias("prefix"),
        F.col("name.suffix").alias("suffix"),
        F.col("name.text").alias("full_name")
    )
    .filter(F.col("name_position").isNotNull())
)

(patient_names_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_patient_names"))

print("Created gold_patient_names")

StatementMeta(, 49fdc724-cb50-4084-9aa3-657047cea021, 4, Finished, Available, Finished, False)

Processing Patient...


Loaded Bronze table: bronze_patient
Processing Encounter...


Loaded Bronze table: bronze_encounter
Processing Observation...


Loaded Bronze table: bronze_observation
Processing Condition...


Loaded Bronze table: bronze_condition


In [ ]:
# Layer validation
for layer, prefix in [("Silver", "silver_"), ("Gold", "gold_")]:
    print(f"--- {layer} ---")
    for resource in RESOURCES:
        table_name = f"{prefix}{resource.lower()}"
        if spark.catalog.tableExists(table_name):
            print(table_name, spark.table(table_name).count(), "rows")
